# The Price Is Right - Week 7 - Day 3 (RL Training)

## Training with Reinforcement Learning (PPO)

In this notebook, we will train the model using Proximal Policy Optimization (PPO) to directly minimize the pricing error.

**Goal**: Refine a standard LLM to become a price prediction expert using Reinforcement Learning.

**Reward Function**: `Reward = -|Predicted Price - True Price|`

### Setup
If using `LITE_MODE=True`, a T4 GPU is sufficient. For full training, use an A100 or L4.

In [ ]:
# 1. Install Libraries
# AGGRESSIVE DEPENDENCY RESOLUTION
# We must ensure numpy<2.0 and compatible torch/transformers versions to avoid binary incompatibility.

import os
import sys

def install_dependencies():
    print("Installing dependencies... This may take a minute.")
    
    # 1. Uninstall critical packages to avoid conflicts
    !pip uninstall -y numpy torch transformers trl peft accelerate bitsandbytes
    
    # 2. Install numpy first (pinned)
    !pip install "numpy<2.0"
    
    # 3. Install Torch (pinned to stable 2.4.1 to match typical Colab drivers and avoid 2.5.0+ issues with old numpy)
    !pip install "torch==2.4.1" "torchvision==0.19.1" "torchaudio==2.4.1" --index-url https://download.pytorch.org/whl/cu121
    
    # 4. Install ecosystem libraries
    # We pin transformers slightly back to be safe with TRL 0.9.6
    !pip install "transformers==4.44.2" "datasets>=2.19.0" "accelerate==0.34.2" "peft==0.12.0" "bitsandbytes==0.43.3" "trl==0.9.6" "wandb"

    print("Dependencies installed.")
    
    # 5. Check if restart is needed (Colab preloads numpy)
    if "google.colab" in sys.modules:
        print("Restarting Colab runtime to apply changes...")
        import IPython
        app = IPython.Application.instance()
        app.kernel.do_shutdown(True)

try:
    import trl
    import numpy
    if numpy.__version__ >= "2.0.0":
        print(f"Detected Numpy {numpy.__version__}. Reinstalling...")
        install_dependencies()
except ImportError:
    install_dependencies()


Installing dependencies... This may take a minute.
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-trans

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 118.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 113.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 55.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 150.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 19.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 43.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124

: 

In [1]:
import os
import re
import torch
import math
import wandb
from tqdm import tqdm
from datetime import datetime
from getpass import getpass
from huggingface_hub import login
import numpy as np

print(f"Numpy Version: {np.__version__}")
if np.__version__ >= '2.0.0':
    raise RuntimeError("Numpy version is >= 2.0.0. Please Restart Session and run the install cell again.")

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from datasets import load_dataset

# Verify versions
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"TRL: {trl.__version__}")

Numpy Version: 1.26.4
PyTorch: 2.4.1+cu121
Transformers: 4.44.2


NameError: name 'trl' is not defined

In [2]:
# 2. Configuration

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price-rl"
HF_USER = "Rodan009" # Update/Verify your HF username if needed

LITE_MODE = True

DATA_USER = "Rodan009"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
    
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# LoRA Config
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 64
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.05

# PPO Config
LEARNING_RATE = 1.41e-5
BATCH_SIZE = 64 if LITE_MODE else 128
MINI_BATCH_SIZE = 8 if LITE_MODE else 16
GRADIENT_ACCUMULATION_STEPS = 1
MAX_NEW_TOKENS = 12
PPO_EPOCHS = 1

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8 # Use bf16 on Ampere+ (A100, L4)

LOG_TO_WANDB = True

In [3]:
# 3. Logins
if 'HF_TOKEN' in os.environ:
    hf_token = os.environ['HF_TOKEN']
else:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except:
        hf_token = getpass("Hugging Face Token: ")
login(hf_token, add_to_git_credential=True)

if LOG_TO_WANDB:
    if 'WANDB_API_KEY' not in os.environ:
        try:
            from google.colab import userdata
            os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
        except:
            pass

    if 'WANDB_API_KEY' not in os.environ:
        os.environ["WANDB_API_KEY"] = getpass("WandB API Key: ")
    wandb.login()
    os.environ["WANDB_PROJECT"] = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rodan009 (rodan009-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# 4. Load Model & Tokenizer

print("Loading Base Model...")
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

print("Applying LoRA...")
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
base_model = get_peft_model(base_model, peft_config)

print("Wrapping with AutoModelForCausalLMWithValueHead...")
# We wrap the PEFT model. TRL handles the value head on top.
model = AutoModelForCausalLMWithValueHead(base_model)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Important for generation

print("Model Loaded Successfully")

Loading Base Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Applying LoRA...
Wrapping with AutoModelForCausalLMWithValueHead...


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Model Loaded Successfully


In [5]:
# 5. Data Processing

def extract_price(text):
    # Extract first float found in text
    if not isinstance(text, str):
        return 0.0
    match = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(match.group()) if match else 0.0

def build_dataset(dataset_name, tokenizer):
    print(f"Loading dataset: {dataset_name}")
    try:
        ds = load_dataset(dataset_name, split="train")
    except Exception as e:
        # Fallback to local if HUB version fails or not found
        print(f"Could not load from hub, trying to synthesize for demo: {e}")
        from datasets import Dataset
        data = [
            {"prompt": "What is the price of a banana?", "completion": "The price is $0.50"},
            {"prompt": "How much for this car?", "completion": "Price: $25000"}
        ] * 10
        ds = Dataset.from_list(data)
    
    def preprocess_function(examples):
        new_examples = {
            "query": [],
            "input_ids": [],
            "true_price": []
        }
        for prompt, completion in zip(examples["prompt"], examples["completion"]):
             new_examples["query"].append(prompt)
             # Tokenize prompt without special tokens to allow generation continuation
             tokenized = tokenizer.encode(prompt, add_special_tokens=False)
             new_examples["input_ids"].append(tokenized)
             new_examples["true_price"].append(extract_price(completion))

        return new_examples

    ds = ds.map(preprocess_function, batched=True, remove_columns=ds.column_names)
    ds = ds.filter(lambda x: len(x["input_ids"]) < 512)
    ds.set_format(type="torch")
    return ds

dataset = build_dataset(DATASET_NAME, tokenizer)
print(f"Dataset loaded: {len(dataset)} examples")
print("Sample:", dataset[0])

Loading dataset: Rodan009/items_prompts_lite


README.md:   0%|          | 0.00/510 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.88M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/246k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/249k [00:00<?, ?B/s]

...

Generating val split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset loaded: 20000 examples
Sample: {'query': 'What does this cost to the nearest dollar?\n\nTitle: Schlage Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half)\nCategory: Door Hardware\nBrand: Schlage\nDescription: Interior half of a two-piece Andover knob with deadbolt in oil rubbed bronze.\nDetails: Includes knob and deadbolt; non-handed knob style; requires F58 to complete handle set; 4" minimum center-to-center door prep; Lifetime Mechanical and Finish Warranty.\n\nPrice is $', 'input_ids': tensor([ 3923,  1587,   420,  2853,   311,   279, 24379, 18160,  1980,  3936,
           25, 50379,   425,  1628,  2017, 29958, 13934,   677,   449, 15371,
        53533,    11, 15895, 13134,  2788, 45967,   320, 86225, 26924,   340,
         6888,    25, 25166, 37865,   198, 28268,    25, 50379,   425,   198,
         5116,    25, 29958,  4376,   315,   264,  1403, 56964,  1628,  2017,
        59672,   449,  5710, 53533,   304,  5707, 67854, 40907,   627,  7955,
          

In [6]:
# 6. Initialize PPO Trainer

def collator(data):
    # Extract true_prices first
    true_prices = [d["true_price"] for d in data]
    
    # Prepare input batch for padding
    # We only care about input_ids and attention_mask (if exists)
    model_inputs = [{k: v for k, v in d.items() if k in ["input_ids", "attention_mask"]} for d in data]
    
    # Pad the inputs
    padding_collator = transformers.DataCollatorWithPadding(tokenizer)
    batch = padding_collator(model_inputs)
    
    # Add true_price back
    batch["true_price"] = torch.tensor(true_prices)
    return batch

config = PPOConfig(
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    remove_unused_columns=False,
    gradient_checkpointing=True
)

# We explicitly pass ref_model=None to let TRL handle the reference model (shared weights)
ppo_trainer = PPOTrainer(
    config=config,
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)
print("PPO Trainer Initialized")

PPO Trainer Initialized


In [7]:
# 7. Training Loop

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": MAX_NEW_TOKENS,
}

if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

save_steps = 20
step_count = 0

print("Starting Training...")

for epoch in range(PPO_EPOCHS):
    print(f"Epoch {epoch+1}/{PPO_EPOCHS}")
    for batch in tqdm(ppo_trainer.dataloader):
        # 1. Prepare Queries
        # PPOTrainer expects a list of tensors for queries
        query_tensors = [t for t in batch["input_ids"]]
        
        # 2. Generate Responses
        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,
            **generation_kwargs
        )
        
        batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]
        
        # 3. Compute Rewards
        rewards = []
        predicted_prices = []
        errors = []
        
        for response, true_price in zip(batch["response"], batch["true_price"]):
            pred_price = extract_price(response)
            predicted_prices.append(pred_price)
            
            # Reward is negative absolute error
            error = abs(pred_price - true_price.item())
            rewards.append(torch.tensor(-error))
            errors.append(error)
            
        # 4. PPO Step
        # Pass tensors directly
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        
        # 5. Logging
        if LOG_TO_WANDB:
            batch_reward = torch.stack(rewards).mean().item()
            avg_error = sum(errors) / len(errors)
            
            log_data = {
                "reward": batch_reward,
                "error": avg_error,
                "mean_true_price": batch["true_price"].float().mean().item(),
                "mean_predicted_price": sum(predicted_prices) / len(predicted_prices),
            }
            log_data.update(stats)
            wandb.log(log_data)
            
        step_count += 1
        if step_count % save_steps == 0:
            print(f"Saving checkpoint at step {step_count}...")
            ppo_trainer.save_pretrained(PROJECT_RUN_NAME)

print("Training Completed.")
ppo_trainer.save_pretrained(PROJECT_RUN_NAME)
print(f"Model saved to {PROJECT_RUN_NAME}")

# Optional: Push to Hub
try:
    ppo_trainer.model.push_to_hub(HUB_MODEL_NAME)
    print(f"Pushed to Hub: {HUB_MODEL_NAME}")
except Exception as e:
    print(f"Could not push to hub: {e}")

if LOG_TO_WANDB:
    wandb.finish()

Starting Training...
Epoch 1/1


  0%|          | 0/312 [00:29<?, ?it/s]


AttributeError: 'AutoModelForCausalLMWithValueHead' object has no attribute 'is_peft_model'

## Conclusion

We have successfully trained a model to price items using PPO. 
Check the WandB dashboard to see the Error rate dropping over time!